In [ ]:
import os
import json
from llama_cpp import Llama

# Configuration
MODEL_PATH = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/gpt-oss-20b_q8.gguf"
INPUT_JSON = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/result_94/TiJian94.json"           
OUTPUT_JSON = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/result_94/results_with_GPT.json"

GPU_LAYERS = -1
N_CTX = 32768               
N_BATCH = 1024
N_UBATCH = 512
N_THREADS = 4
MAIN_GPU = 0
MAX_TOKENS = 4096
TEMPERATURE = 0.1
TOP_P = 0.8
SAVE_EVERY = 10

STOP_TOKENS = ["<|im_end|>", "<|endoftext|>"]  # Common stop tokens for GPT-OSS

# 1. Load model
if not os.path.exists(MODEL_PATH):
    print(f"Error: GGUF file not found at {MODEL_PATH}")
    exit(1)

print("Loading GPT-OSS model on GPU...")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    n_batch=N_BATCH,
    n_ubatch=N_UBATCH,
    n_threads=N_THREADS,
    n_gpu_layers=GPU_LAYERS,
    main_gpu=MAIN_GPU,
    flash_attn=True,
    f16_kv=True,
    use_mmap=True,
    use_mlock=False,
    offload_kqv=True,
    verbose=False,
)
print("Model loaded successfully.")

# 2. Read data
with open(INPUT_JSON, 'r', encoding='utf-8') as f:
    data = json.load(f)
print(f"Total records: {len(data)}")

# 3. Batch processing
results = []
for idx, item in enumerate(data):
    instruction = item.get("instruction", "")
    input_text = item.get("input", "")

    # Construct messages: same format as interacting with GPT-OSS
    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": input_text}
    ]

    generated_text = ""
    try:
        response = llm.create_chat_completion(
            messages=messages,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS,
            stream=False,
            stop=STOP_TOKENS,
        )
        generated_text = response["choices"][0]["message"]["content"]
    except Exception as e:
        print(f"Error on item {idx+1}: {e}")
        print(f"Failed item: instruction={instruction[:100]}..., input={input_text[:100]}...")

    # Save results
    new_item = item.copy()
    new_item["output_GPTOSS"] = generated_text
    results.append(new_item)

    # Save checkpoint every 10 records
    if (idx + 1) % 10 == 0:
        with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"Saved {idx+1} records.")

# 4. Final save
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"All done! Results saved to {OUTPUT_JSON}")